In [2]:
from youtube_transcript_api import YouTubeTranscriptApi
from dotenv import load_dotenv
import os

In [3]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
print("API key loaded!" if api_key else "API key not found - check your .env file")

API key loaded!


In [4]:
def get_video_id(url):
    if "v=" in url:
        video_id = url.split("v=")[1]
        video_id = video_id.split("&")[0]
        return video_id
    elif "youtu.be/" in url:
        video_id = url.split("youtu.be/")[1]
        video_id = video_id.split("?")[0]
        return video_id
    else:
        print("Invalid URL")
        return None

In [5]:
print(get_video_id("https://www.youtube.com/watch?v=aircAruvnKk&t=120"))

aircAruvnKk


In [8]:
def get_transcript(url):
    video_id = get_video_id(url)
    if not video_id:
        print("Invalid Youtube URL")
        return None
    try:
        ytt_api = YouTubeTranscriptApi()
        transcript = ytt_api.fetch(video_id)
        return transcript
    except Exception as e:
        print(f"Error fetching transcript: {e}")
        return None

In [9]:
url = "https://www.youtube.com/watch?v=aircAruvnKk"
transcript = get_transcript(url)

if transcript:
    for entry in transcript[:5]:
        print(entry)
    print(f"\nTotal segments: {len(transcript)}")

FetchedTranscriptSnippet(text='This is a 3.', start=4.22, duration=1.18)
FetchedTranscriptSnippet(text="It's sloppily written and rendered at an extremely low resolution of 28x28 pixels,", start=6.06, duration=4.653)
FetchedTranscriptSnippet(text='but your brain has no trouble recognizing it as a 3.', start=10.713, duration=3.007)
FetchedTranscriptSnippet(text='And I want you to take a moment to appreciate how', start=14.34, duration=2.219)
FetchedTranscriptSnippet(text='crazy it is that brains can do this so effortlessly.', start=16.559, duration=2.401)

Total segments: 286


In [12]:
def chunk_transcript(transcript, chunk_duration=120):
    chunks = []
    current_text = ""
    chunk_start = transcript[0].start

    for segment in transcript:
        if segment.start - chunk_start >= chunk_duration:
            chunks.append({
                "text": current_text.strip(),
                "start": chunk_start,
                "end": segment.start
            })
            current_text = segment.text + " "
            chunk_start = segment.start
        else:
            current_text += segment.text + " "

    # Don't forget the last chunk that's still being built
    if current_text:
        chunks.append({
            "text": current_text.strip(),
            "start": chunk_start,
            "end": transcript[-1].start
        })

    return chunks

In [13]:
chunks = chunk_transcript(transcript)
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} | {chunk['start']:.0f}s - {chunk['end']:.0f}s")
    print(chunk['text'][:100] + "...")
    print()

Chunk 1 | 4s - 125s
This is a 3. It's sloppily written and rendered at an extremely low resolution of 28x28 pixels, but ...

Chunk 2 | 125s - 246s
There are many many variants of neural networks, and in recent years there's been sort of a boom in ...

Chunk 3 | 246s - 368s
which for the time being should just be a giant question mark for how on earth this process of recog...

Chunk 4 | 368s - 488s
Now in a perfect world, we might hope that each neuron in the second to last layer corresponds with ...

Chunk 5 | 488s - 610s
Parsing speech, for example, involves taking raw audio and picking out distinct sounds, which combin...

Chunk 6 | 610s - 733s
are bright but the surrounding pixels are darker. When you compute a weighted sum like this, you mig...

Chunk 7 | 733s - 854s
The connections between the other layers also have a bunch of weights and biases associated with the...

Chunk 8 | 854s - 975s
By the way, so much of machine learning just comes down to having a good grasp of linear al